<a href="https://colab.research.google.com/github/Angappan95/async-practice/blob/main/02_async_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Invoke three providers concurrently using httpx.AsyncClient.

2. Limit downstream concurrency using asyncio.Semaphore.

3. Set timeouts and implement bounded retries for transient failures.
4. Continue with successful results when a provider fails.

5. Deduplicate documents using doc_id

6. Sort documents by score and return the top five.

7. Record individual provider latency and overall response time.

8. Reuse HTTP connections with FastAPI lifespan.

9. Return HTTP 503 when all providers fail.

10. Load-test the service with 100 concurrent requests.

In [ ]:
from fastapi import FastAPI
from time import perf_counter
import asyncio
import random
import httpx

app = FastAPI()

@app.get("/provider")
async def get_provider():
    await asyncio.sleep(random.randint(1, 10))
    return {
        "document_id": f"doc_{random.randint(1, 5)}",
        "document_score": f"{random.random():.2f}"
    }

In [ ]:
import nest_asyncio
import uvicorn
from threading import Thread

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

nest_asyncio.apply()

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

In [ ]:
import httpx

async def fetch_from_provider():
    async with httpx.AsyncClient() as client:
        try:
            response = await client.get(f"http://127.0.0.1:8000/provider", timeout=8)
            response.raise_for_status()
            print(response.json())
        except httpx.TimeoutException:
            print("Response Timedout")


await fetch_from_provider()

INFO:     127.0.0.1:35588 - "GET /provider HTTP/1.1" 200 OK
{'document_id': 'doc_5', 'document_score': '0.11'}


In [ ]:
import httpx
import asyncio

semaphore = asyncio.Semaphore(3)

async def fetch_from_provider(client):
    async with semaphore:
        try:
            response = await client.get(f"http://127.0.0.1:8000/provider", timeout=8)
            response.raise_for_status()
            # print(response.json())
            return response.json()
        except httpx.TimeoutException:
            return {
                "status": "Timed out"
            }

client = httpx.AsyncClient()

await fetch_from_provider(client)
await fetch_from_provider(client)

INFO:     127.0.0.1:53712 - "GET /provider HTTP/1.1" 200 OK
INFO:     127.0.0.1:53712 - "GET /provider HTTP/1.1" 200 OK


{'document_id': 'doc_4', 'document_score': '0.84'}

In [ ]:
import httpx
import asyncio
import json

async def fetch_data():
    async with httpx.AsyncClient() as client:
        start = perf_counter()
        data = await asyncio.gather(
            *(fetch_from_provider(client) for _ in range(1, 10))
        )
        print("Overall response time to collect data from 10 provider call {}", perf_counter()-start)

    print(json.dumps(data, indent=2))
    result = []
    seen = set()
    for item in data:
        if 'document_id' in item and item['document_id'] not in seen:
            result.append(item)
            seen.add(item['document_id'])

    result = sorted(
        result, key=lambda x: float(x['document_score']), reverse=True
        )

    print(json.dumps(result, indent=2))

await fetch_data()

INFO:     127.0.0.1:58152 - "GET /provider HTTP/1.1" 200 OK
INFO:     127.0.0.1:58150 - "GET /provider HTTP/1.1" 200 OK
INFO:     127.0.0.1:58150 - "GET /provider HTTP/1.1" 200 OK
INFO:     127.0.0.1:54092 - "GET /provider HTTP/1.1" 200 OK
INFO:     127.0.0.1:54092 - "GET /provider HTTP/1.1" 200 OK
INFO:     127.0.0.1:58150 - "GET /provider HTTP/1.1" 200 OK
INFO:     127.0.0.1:35984 - "GET /provider HTTP/1.1" 200 OK
Overall response time to collect data from 10 provider call {} 18.026398656998936
[
  {
    "document_id": "doc_5",
    "document_score": "0.46"
  },
  {
    "document_id": "doc_4",
    "document_score": "0.71"
  },
  {
    "status": "Timed out"
  },
  {
    "status": "Timed out"
  },
  {
    "document_id": "doc_1",
    "document_score": "0.12"
  },
  {
    "document_id": "doc_4",
    "document_score": "0.53"
  },
  {
    "document_id": "doc_2",
    "document_score": "0.42"
  },
  {
    "document_id": "doc_1",
    "document_score": "0.61"
  },
  {
    "document_id": "doc_4"